# Noise-Controlled NCA Training

This notebook trains a single NCA model to produce **2 or 3 different textures** based on the noise level:
- **2 textures**: noise=0 → Texture 1, noise=high → Texture 2
- **3 textures**: noise=0 → Texture 1, noise=mid → Texture 2, noise=high → Texture 3

The key mechanism: during training, different pool elements receive different noise levels and are trained against different target textures. The network learns to associate noise level with texture output.

In [ ]:
# @title Imports and Notebook Utilities
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Loss Function (Sliced OT)
import torch.nn.functional as F

# Load VGG for feature extraction
vgg = models.vgg16(weights='IMAGENET1K_V1').features

def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_sliced_ot_loss(target_img):
    """Create a Sliced OT loss function for a target image."""
    with torch.no_grad():
        yy = calc_styles_vgg(target_img, vgg)
    def loss_f(imgs):
        xx = calc_styles_vgg(imgs, vgg)
        return sum(ot_loss(x, y) for x, y in zip(xx, yy))
    return loss_f

print("VGG loaded and Sliced OT loss function defined.")

In [ ]:
#@title Load Target Images (2 or 3 textures)
num_textures = 2  #@param [2, 3] {type: "raw"}

from google.colab import files

target_imgs = []
texture_paths = []

for i in range(num_textures):
    if i == 0:
        label = "LOW noise (0.0)"
    elif i == 1 and num_textures == 2:
        label = "HIGH noise"
    elif i == 1:
        label = "MEDIUM noise"
    else:
        label = "HIGH noise"

    print(f"\nUpload TEXTURE {i+1} (appears at {label}):")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    texture_paths.append(path)

    img = imread(io.BytesIO(uploaded[path]), max_size=128)
    target_imgs.append(img)
    print(f"Texture {i+1}:")
    imshow(img)

# Create loss functions for each target
loss_fns = []
for i, img in enumerate(target_imgs):
    tensor = torch.tensor(img).permute(2, 0, 1).unsqueeze(0)
    loss_fn = create_sliced_ot_loss(tensor)
    loss_fns.append(loss_fn)
    print(f"Created loss function for texture {i+1}")

print(f"\n\u2713 Created {num_textures} loss functions")
print(f"  Texture indices: {list(range(num_textures))}")

In [ ]:
#@title NoiseNCA Architecture
def depthwise_conv(x, filters):
    """filters: [filter_n, h, w]"""
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)

def merge_lap(z):
    # Merge lap_x and lap_y into a single laplacian filter
    b, c, h, w = z.shape  # [b, 5 * chn, h, w]
    z = torch.stack([
        z[:, ::5],
        z[:, 1::5],
        z[:, 2::5],
        z[:, 3::5] + z[:, 4::5]
    ], dim=2)  # [b, chn, 4, h, w]
    return z.reshape(b, -1, h, w)  # [b, 4 * chn, h, w]


class NoiseNCA(torch.nn.Module):
    def __init__(self, chn=12, fc_dim=96, noise_level=1.0):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
            lap_x = torch.tensor([[0.5, 0.0, 0.5], [2.0, -6.0, 2.0], [0.5, 0.0, 0.5]])
            self.filters = torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T])

    def perception(self, s, dx=1.0, dy=1.0):
        z = depthwise_conv(s, self.filters)  # [b, 5 * chn, h, w]
        if isinstance(dx, float) and dx == 1.0 and isinstance(dy, float) == 1.0:
            return merge_lap(z)

        if not isinstance(dx, torch.Tensor) or dx.ndim != 3:
            dx = torch.tensor([dx], device=s.device)[:, None, None]
        if not isinstance(dy, torch.Tensor) or dy.ndim != 3:
            dy = torch.tensor([dy], device=s.device)[:, None, None]

        scale = 1.0 / torch.stack([torch.ones_like(dx), dx, dy, dx ** 2, dy ** 2], dim=1)
        scale = torch.tile(scale, (1, self.chn, 1, 1))
        z = z * scale
        return merge_lap(z)

    def forward(self, s, dx=1.0, dy=1.0, dt=1.0, noise=None):
        if noise is not None:
            # Support both scalar and per-batch noise
            if isinstance(noise, torch.Tensor) and noise.ndim >= 1:
                # Per-batch noise: reshape to [b, 1, 1, 1] for broadcasting
                if noise.ndim == 1:
                    noise = noise.reshape(-1, 1, 1, 1)
            s = s + torch.randn_like(s) * noise
        z = self.perception(s, dx, dy)
        delta_s = self.w2(torch.relu(self.w1(z)))
        return s + delta_s * dt

    def seed(self, n, h=128, w=128):
        return (torch.rand(n, self.chn, h, w) - 0.5) * self.noise_level


def to_rgb(s):
    return s[..., :3, :, :] + 0.5


param_n = sum(p.numel() for p in NoiseNCA().parameters())
print('NoiseNCA param count:', param_n)

In [ ]:
#@title Setup Training
import os
import glob
from google.colab import files

# Training hyperparameters - noise levels for each texture
# Automatically space noise levels from 0 to max_noise
max_noise = 0.04  #@param {type: "number"}
noise_levels = [i * max_noise / (num_textures - 1) for i in range(num_textures)]
print(f"Noise levels for {num_textures} textures: {noise_levels}")

# Check for existing checkpoints
checkpoint_files = glob.glob('noise_controlled_*.pt')

if checkpoint_files:
    print(f"\nFound {len(checkpoint_files)} checkpoint file(s):")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")
    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()
    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Initialize model
model = NoiseNCA()

if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iter = checkpoint.get('iteration', 0) + 1
    loss_log = checkpoint.get('loss_log', [])
    pool = checkpoint.get('pool', model.seed(256))
    print(f"Resumed from iteration {start_iter - 1}")
else:
    start_iter = 0
    loss_log = []
    with torch.no_grad():
        pool = model.seed(256)
    print("Starting fresh training")

# Optimizer with adaptive LR
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.3, patience=500,
    threshold=0.01, threshold_mode='rel', min_lr=1e-6
)

if checkpoint and 'optimizer_state_dict' in checkpoint:
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
if checkpoint and 'scheduler_state_dict' in checkpoint:
    lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])

print(f"\nPool shape: {pool.shape}")
print(f"Training {num_textures} textures with noise levels: {noise_levels}")

In [ ]:
#@title Training Loop {vertical-output: true}

num_iterations = 10000  #@param {type: "integer"}
batch_size = 4  #@param {type: "integer"}

try:
    for i in range(start_iter, start_iter + num_iterations):
        with torch.no_grad():
            # Sample batch indices from pool
            batch_idx = np.random.choice(len(pool), batch_size, replace=False)
            s = pool[batch_idx]

            # Inject fresh seed periodically
            if i % 32 == 0:
                s[:1] = model.seed(1)

            # KEY: Map batch indices to texture indices (0, 1, or 2)
            texture_indices = batch_idx % num_textures

            # Get noise level for each batch element based on its target texture
            batch_noise = torch.tensor(
                [noise_levels[t] for t in texture_indices],
                device='cuda', dtype=torch.float32
            )

        # Run forward steps with noise-controlled texture selection
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, noise=batch_noise)

        # Compute loss - each batch element uses its corresponding target
        overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()

        # Sum losses from appropriate loss functions
        loss = overflow_loss
        rgb = to_rgb(s)
        for b in range(batch_size):
            t_idx = texture_indices[b]
            loss = loss + loss_fns[t_idx](rgb[b:b+1])

        # Backward and optimize
        with torch.no_grad():
            loss.backward()
            for p in model.parameters():
                p.grad /= (p.grad.norm() + 1e-8)
            opt.step()
            opt.zero_grad()
            lr_sched.step(loss)
            pool[batch_idx] = s.detach()
            loss_log.append(loss.item())

            # Display progress
            if i % 10 == 0:
                lr = opt.param_groups[0]['lr']
                display(Markdown(f"iter: {i}, loss: {loss.item():.2e}, lr: {lr:.2e}"), display_id='stats')

            # Visualize
            if i % 20 == 0:
                # Show loss plot and current batch
                pl.figure(figsize=(12, 3))
                
                pl.subplot(1, 2, 1)
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.title('Loss')
                pl.xlabel('Iteration')

                # Show current batch with texture labels
                pl.subplot(1, 2, 2)
                imgs = rgb.permute(0, 2, 3, 1).cpu().numpy()
                labels = [f'T{t+1}' for t in texture_indices]
                pl.imshow(np.hstack(imgs))
                pl.title(' | '.join(labels))
                pl.axis('off')

                pl.tight_layout()
                imshow(grab_plot(), id='progress')

            # Save checkpoint
            if i % 1000 == 0 and i > start_iter:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'scheduler_state_dict': lr_sched.state_dict(),
                    'loss_log': loss_log,
                    'pool': pool,
                    'noise_levels': noise_levels,
                    'num_textures': num_textures,
                }
                torch.save(checkpoint, f'noise_controlled_iter_{i}.pt')
                print(f"\nCheckpoint saved at iteration {i}")

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
        'noise_levels': noise_levels,
        'num_textures': num_textures,
    }
    torch.save(checkpoint, f'noise_controlled_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved!')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Test Noise Control

# Generate samples across the noise range
max_test_noise = max(noise_levels) * 1.5  # Go beyond training range
test_noise_levels = np.linspace(0, max_test_noise, 8)

with torch.no_grad():
    results = []
    for nl in test_noise_levels:
        s = model.seed(1)
        for _ in range(64):
            s = model(s, noise=nl)
        results.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())

    # Display
    fig, axes = pl.subplots(1, len(test_noise_levels), figsize=(16, 2))
    for ax, img, nl in zip(axes, results, test_noise_levels):
        ax.imshow(img)
        ax.set_title(f'{nl:.3f}')
        ax.axis('off')

    # Add markers for training noise levels
    noise_str = ', '.join([f'{n:.3f}' for n in noise_levels])
    pl.suptitle(f'Noise Level \u2192 Texture Transition\nTraining levels: [{noise_str}]')
    pl.tight_layout()
    imshow(grab_plot())

# Show target textures for comparison
print("\nTarget textures for reference:")
fig, axes = pl.subplots(1, num_textures, figsize=(4 * num_textures, 4))
if num_textures == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, target_imgs)):
    ax.imshow(img)
    ax.set_title(f'Texture {i+1} (noise={noise_levels[i]:.3f})')
    ax.axis('off')
pl.tight_layout()
imshow(grab_plot())

In [ ]:
#@title Save Model Weights

# Save for demo
weights_file = 'noise_controlled_weights.pt'

# Include training params for conversion script
state_dict = model.state_dict()
state_dict['_training_params'] = {
    'noise_levels': noise_levels,
    'num_textures': num_textures,
    'model_type': 'nca',
    'texture_paths': texture_paths,
}

torch.save(state_dict, weights_file)
print(f"Saved: {weights_file}")
print(f"  Textures: {num_textures}")
print(f"  Noise levels: {noise_levels}")

# Also save full checkpoint
checkpoint_file = 'noise_controlled_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
    'noise_levels': noise_levels,
    'num_textures': num_textures,
}
torch.save(checkpoint, checkpoint_file)
print(f"\nFull checkpoint saved: {checkpoint_file}")

# Download
from google.colab import files
files.download(weights_file)